In [ ]:
import datetime
import threading
import time
import collections
import json
import xmlrpc.client
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler

UPSTREAM_URL      = "http://127.0.0.1:8009"     # xmlrpc_server
STATS_URL         = "http://127.0.0.1:8018"     # xmlrpc_stats2_server
PROXY_ADDR        = ("127.0.0.1", 8008)
RATE_LIMIT_N      = 5
RATE_LIMIT_T      = 5.0
FALLBACK_LOG      = "proxy_fallback_log.jsonl"  # если stats недоступен

upstream = xmlrpc.client.ServerProxy(UPSTREAM_URL, allow_none=True)
stats = xmlrpc.client.ServerProxy(STATS_URL, allow_none=True)


class RateLimiter:
    def __init__(self, n, t):
        self.n, self.t = n, t
        self.calls = collections.deque()     # монотонные отметки времени
        self.lock = threading.Lock()

    def acquire(self) -> bool:
        """True — разрешено; False — лимит исчерпан."""
        now = time.monotonic()
        with self.lock:
            while self.calls and now - self.calls[0] > self.t:
                self.calls.popleft()
            if len(self.calls) >= self.n:
                return False
            self.calls.append(now)
            return True

    def wait_time(self) -> float:
        with self.lock:
            if not self.calls:
                return 0.0
            return max(0.0, self.t - (time.monotonic() - self.calls[0]))

limiter = RateLimiter(RATE_LIMIT_N, RATE_LIMIT_T)


def _now_iso():
    return datetime.datetime.now().isoformat(timespec="microseconds")

def _fallback(event):
    with open(FALLBACK_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False) + "\n")

def send_event(event_type, method=None, duration=None,
               status=None, client=None, ts_event=None):
    ts = ts_event or _now_iso()
    event = {
        "ts_event":   ts,
        "event_type": event_type,
        "method":     method,
        "duration":   duration,
        "status":     status,
        "client":     client,
    }
    try:
        stats.log_event(ts, event_type, method, duration, status, client)
    except Exception as e:
        print("[stats-fallback]", type(e).__name__, ":", e)
        _fallback(event)


def _dispatch(method, params, client):
    """
    Возвращает (result, error) — повторяет поведение xmlrpc.client:
    результат либо Fault. Логирует operation / too_many_requests.
    """

    if not limiter.acquire():
        wait = limiter.wait_time()
        ts = _now_iso()
        send_event("too_many_requests", method=method,
                   duration=None, status="rate_limited",
                   client=client, ts_event=ts)
        msg = (f"Rate limit exceeded: {RATE_LIMIT_N} requests / "
               f"{RATE_LIMIT_T}s. Retry after {wait:.2f}s.")
        return xmlrpc.client.Fault(-32000, msg)


    ts_start = _now_iso()
    t0 = time.perf_counter()
    status = "ok"
    try:
        result = method(*params)
    except xmlrpc.client.Fault as f:
        status = "error"
        result = f
    except Exception as e:
        status = "error"
        result = xmlrpc.client.Fault(-32500, f"{type(e).__name__}: {e}")
    duration = time.perf_counter() - t0

    send_event("operation", method=getattr(method, "_rpc_name", None) or str(method),
               duration=duration, status=status,
               client=client, ts_event=ts_start)
    return result


class _Method:
    """Прокси-обёртка над конкретным методом upstream."""
    def __init__(self, name, proxy):
        self._rpc_name = name
        self._proxy = proxy

    def __call__(self, *params):
        return self._proxy._invoke(self._rpc_name, params)


class Proxy:
    """XMLRPC-прокси: динамически пробрасывает любые методы upstream."""
    def __init__(self):
        self._client = xmlrpc.client.ServerProxy(UPSTREAM_URL, allow_none=True)

    def _invoke(self, name, params):

        if not limiter.acquire():
            wait = limiter.wait_time()
            send_event("too_many_requests", method=name,
                       status="rate_limited", client="xmlrpc",
                       ts_event=_now_iso())
            raise xmlrpc.client.Fault(
                -32000,
                f"Rate limit exceeded: {RATE_LIMIT_N}/{RATE_LIMIT_T}s. "
                f"Retry after {wait:.2f}s."
            )

        ts_start = _now_iso()
        t0 = time.perf_counter()
        status = "ok"
        try:
            remote = getattr(self._client, name)
            result = remote(*params)
        except xmlrpc.client.Fault:
            status = "error"
            raise
        except Exception as e:
            status = "error"
            raise xmlrpc.client.Fault(-32500, f"{type(e).__name__}: {e}")
        finally:
            duration = time.perf_counter() - t0
            send_event("operation", method=name, duration=duration,
                       status=status, client="xmlrpc", ts_event=ts_start)
        return result

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        return _Method(name, self)


_proxy = Proxy()


def make_forwarder(name):
    def _f(*params):
        return _proxy._invoke(name, params)
    _f.__name__ = name
    return _f


FORWARDED = [
    "ping", "now", "type",
    "sum", "pow",
    "send_back_binary", "color_inversion", "binarization_percent",
    "black_list_check", "black_list_check_fio",
]


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

if __name__ == "__main__":
    server = SimpleXMLRPCServer(
        PROXY_ADDR,
        requestHandler=RequestHandler,
        allow_none=True,
        logRequests=False,
    )

    for name in FORWARDED:
        server.register_function(make_forwarder(name), name)


    def proxy_stats():
        with limiter.lock:
            return {
                "rate_limit_n": RATE_LIMIT_N,
                "rate_limit_t": RATE_LIMIT_T,
                "calls_in_window": len(limiter.calls),
                "upstream": UPSTREAM_URL,
                "stats": STATS_URL,
            }
    server.register_function(proxy_stats, "proxy_stats")

    stats_probe = xmlrpc.client.ServerProxy("http://127.0.0.1:8018", allow_none=True)


    print(f"Proxy server on http://{PROXY_ADDR[0]}:{PROXY_ADDR[1]}")
    print(f"  upstream: {UPSTREAM_URL}")
    print(f"  stats   : {STATS_URL}")
    print(f"  rate    : {RATE_LIMIT_N} / {RATE_LIMIT_T}s")
    server.serve_forever()

Proxy server on http://127.0.0.1:8008
  upstream: http://127.0.0.1:8009
  stats   : http://127.0.0.1:8018
  rate    : 10 / 5.0s
